In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)

CATALOG = "streaming_demo"
SCHEMA = "trades"

SOURCE = "/Volumes/streaming_demo/trades/landing/landing"

CHECKPOINT = (
    "/Volumes/streaming_demo/trades/landing/checkpoints"
)

RAW_TABLE = (
    f"{CATALOG}.{SCHEMA}.raw_trade_events"
)

schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("event_ts", StringType(), True),
    StructField("trade_id", StringType(), True),
    StructField("account_id", StringType(), True),
    StructField("symbol", StringType(), True),
    StructField("side", StringType(), True),
    StructField("quantity", DoubleType(), True),
    StructField("price", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("event_type", StringType(), True)
])

raw = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option(
            "cloudFiles.schemaLocation",
            f"{CHECKPOINT}/schema"
        )
        .schema(schema)
        .load(SOURCE)
        .withColumn(
            "_ingested_at",
            F.current_timestamp()
        )
        .withColumn(
            "_source_file",
            F.col("_metadata.file_path")
        )
)

query = (
    raw.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            f"{CHECKPOINT}/raw"
        )
        .trigger(availableNow=True)
        .toTable(RAW_TABLE)
)

query.awaitTermination()
